# 01 — Embedding Fine-Tuning (multilingual-e5-large + hard negatives)

This notebook fine-tunes `intfloat/multilingual-e5-large` for the Turkish legal domain using a contrastive objective with mined hard negatives.

**Training data.** 3,026 synthetic `(question, article)` pairs, each augmented with 5 BM25-mined hard negatives, yielding ~15K effective training tuples.

**Objective.** `CachedMultipleNegativesRankingLoss` — in-batch negatives combined with the explicit hard negatives. The cached variant is used to keep VRAM under the T4's 16 GB ceiling for long passages.

**Inputs (Kaggle).**
- `embed_triplets.jsonl` — anchor / positive / negatives in JSONL format (~15 MB)

**Output.** `e5-large-tr-legal/` (full fine-tuned `SentenceTransformer`).

**Wall time.** ~2–3 hours on a single T4.

---

**v2 (2026-05-22):** retrained on cleaned corpus (37,927 unique maddes, PDF-tail noise removed, gold-positive isolation enforced). Triplets: `embed_triplets_v3.jsonl` (2,861 anchors × 10 hard negatives, mined from BM25∪dense top-100 with 1.5× same-kanun boost). Batch 16→64, epochs 2→3, LR 2e-5→3e-5 cosine. Output: `e5-large-tr-legal-v2`.

## Setup

In [ ]:
!pip install -q sentence-transformers==3.3.1 peft transformers accelerate datasets

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # DataParallel'ı tetiklemesin — tek GPU yeterli, DP stuck oluyor
os.environ['WANDB_DISABLED'] = 'true'

import json, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

INPUT_DIR = Path('/kaggle/input/datasets/hasanemreusta/turkish-legal-rag-finetune')
TRIPLETS = INPUT_DIR / 'embed_triplets_v3.jsonl'  # v3: cleaned corpus, isolation-clean anchors, 10 hard negs
OUTPUT_DIR = Path('/kaggle/working/e5-large-tr-legal-v2')

BASE_MODEL = 'intfloat/multilingual-e5-large'
BATCH_SIZE = 64    # v1: 16 — larger effective batch via CachedMNRL (mini_batch=8 keeps VRAM ~same)
EPOCHS = 3         # v1: 2
LR = 3e-5          # v1: 2e-5 — slightly higher, paired with cosine schedule
WARMUP_RATIO = 0.1
SCHEDULER = 'warmupcosine'  # v1: warmuplinear (default)

print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())


## Data preparation

Each line of `embed_triplets.jsonl` has the shape:

```json
{"anchor": "...", "positive": "...", "negatives": ["n1", "n2", ...]}
```

E5 expects `"query: "` and `"passage: "` prefixes for asymmetric retrieval; both are added at construction time.

In [ ]:
rows = []
with TRIPLETS.open(encoding='utf-8') as f:
    for line in f:
        rows.append(json.loads(line))
print(f'Toplam triplet: {len(rows)}')
print('Örnek anchor:', rows[0]['anchor'][:100])
print('Örnek positive:', rows[0]['positive'][:100])
print('Negative sayısı:', len(rows[0]['negatives']))

In [ ]:
# Convert to InputExample with hard negatives
# MultipleNegativesRankingLoss expects: anchor, positive, [neg1, neg2, ...]
examples = []
for r in rows:
    anchor = 'query: ' + r['anchor']
    positive = 'passage: ' + r['positive']
    negatives = ['passage: ' + n for n in r['negatives']]
    # InputExample with multiple negatives
    examples.append(InputExample(texts=[anchor, positive] + negatives))
print(f'InputExample sayısı: {len(examples)}')

## Model loading and training

A note on DataParallel: `sentence-transformers` does not handle multi-GPU `DataParallel` cleanly for this loss — training stalls indefinitely. We restrict the visible device to a single GPU (`CUDA_VISIBLE_DEVICES=0`).

In [ ]:
model = SentenceTransformer(BASE_MODEL, device='cuda')
# DataParallel KULLANMA — sentence-transformers DP'de stuck oluyor (test edildi, 4h hiç ilerleme yok)
# CUDA_VISIBLE_DEVICES='0' ile zaten 2. GPU gizlendi

train_loader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
# Cached version VRAM tasarrufu yapar — uzun anchor+positives için kritik
loss = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=8)

In [ ]:
model.fit(
    train_objectives=[(train_loader, loss)],
    epochs=EPOCHS,
    warmup_steps=int(WARMUP_RATIO * len(train_loader) * EPOCHS),
    optimizer_params={'lr': LR},
    scheduler=SCHEDULER,  # v2: cosine decay (smoother LR tail than warmuplinear)
    use_amp=True,
    show_progress_bar=True,
    output_path=str(OUTPUT_DIR),
    save_best_model=False,
    checkpoint_path=str(OUTPUT_DIR / 'checkpoints'),
    checkpoint_save_steps=100,  # daha sık checkpoint, ilerleme görünür
    checkpoint_save_total_limit=2,
)
print('Done. Saved to:', OUTPUT_DIR)


## Smoke test

Reload from disk and verify that semantically aligned (query, passage) pairs receive higher similarity than unrelated ones.

In [ ]:
# Reload fresh and test
tuned = SentenceTransformer(str(OUTPUT_DIR), device='cuda')

queries = [
    'query: Cumhurbaşkanına hakaret cezası nedir?',
    'query: Eşim 5 yıldır kayıp, boşanabilir miyim?',
]
passages = [
    'passage: Cumhurbaşkanına hakaret eden kişi bir yıldan dört yıla kadar hapis cezası alır.',
    'passage: Gaiplik halinde mahkeme kararıyla evlilik feshedilebilir.',
    'passage: Vergi mükellefi defter tutmakla yükümlüdür.',
]
q_emb = tuned.encode(queries, normalize_embeddings=True)
p_emb = tuned.encode(passages, normalize_embeddings=True)
sims = q_emb @ p_emb.T
print('Similarity matrix:')
print(sims)
# Beklenen: diagonal (0,0) ve (1,1) en yüksek

## Export

Download `/kaggle/working/e5-large-tr-legal-v2/` from the Kaggle Output panel and place it under `data/models/e5-large-tr-legal-v2/` in the local repo. Then rebuild the FAISS index:

```powershell
python -m src.retrieval.build_index `
  --corpus data/corpus/mevzuat_full_normalized.jsonl `
  --embed-model data/models/e5-large-tr-legal-v2 `
  --output-dir data/index_full --skip-bm25
```
